# PubChem global-retrieval figures (scaffold split)

Same analysis as `fig_retrieval_results_pubchem_global.ipynb`, but for
**scaffold-split** trained checkpoints (ICICLE `entropy_scaffold_s1`, NEIMS
`neims_scaffold_s1`, MassFormer `massformer_scaffold_s1`) instead of
random-split. Scaffold is the out-of-distribution generalization setting
(test-set scaffolds absent from training) -- expect substantially worse
absolute retrieval numbers than the random-split notebook, consistent with
the paper's framing.

No RI-restricted tracks here: the AIRI RI eval TSV used elsewhere
(`ri_dataset_random_split_no_xeno_aas.tsv`) is built from the **random**
split's test set, so RI-based filtering/ranking would conflate two
different splits. Only MW-global and heavy-atom-global filter tracks are
included (candidate selection only, ranked by cosine similarity -- no RI
involved).

Figures are written to a separate output path (`figures/retrieval_pubchem_global_scaffold`)
so the random-split figures in `figures/retrieval_pubchem_global/` are untouched.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import mark_inset, zoomed_inset_axes
import pandas as pd

from icicle.utils.visualization.eval_plots import model_color
from icicle.utils.visualization.style import (
    get_palette,
    make_fig,
    save_fig,
    set_style,
)

set_style("manuscript")

RESULTS = Path("/home/magled/icicle-dev/results")
OUTPUT_DIR = Path("figures/retrieval_pubchem_global_scaffold")
K_VALUES = [1, 2, 3, 4, 5, 10, 15, 20, 30, 40, 50]
SUMMARY_K_VALUES = [1, 5, 10, 20, 50]
MODES = ["autofail"]
MODE_DISPLAY_NAMES = {
    "inject": "Injected (true spectrum guaranteed in pool)",
    "autofail": "As-scanned (no injection)",
}

## Configuration

`MODEL_PATHS` maps model label to its `retrieval_global_per_query.tsv`
(one row per query, columns `rank_inject_cosine` / `rank_autofail_cosine`).
Unlike the random-split notebook, this is the direct full-test-set file --
the scaffold "all" runs were launched without an `ri_types` restriction, so
no separate StdNP-subset file exists.

In [ ]:
MODEL_PATHS = {
    "ICICLE": RESULTS
    / "pubchem_retrieval_eval_icicle_scaffold_s1"
    / "retrieval_global_per_query.tsv",
    "NEIMS": RESULTS
    / "pubchem_retrieval_eval_neims_scaffold_s1"
    / "retrieval_global_per_query.tsv",
    "MassFormer": RESULTS
    / "pubchem_retrieval_eval_massformer_scaffold_s1"
    / "retrieval_global_per_query.tsv",
}

## Load

One row per query already (rank of the true molecule directly) -- no
decoy rows to filter, unlike the formula/RI retrieval CSVs.

In [ ]:
model_dfs = {}
for label, path in MODEL_PATHS.items():
    if path.exists():
        model_dfs[label] = pd.read_csv(path, sep="\t")
        print(f"{label}: {len(model_dfs[label])} queries loaded")
    else:
        print(f"{label}: {path} not found, skipping")

## Top-k accuracy curves

One figure per ranking mode. All models on the same axes.

In [ ]:
def topk_accuracy_curve(df: pd.DataFrame, rank_col: str, k_values: list[int]):
    """Top-k accuracy (%) at each k, single run (no CI)."""
    ranks = df[rank_col].dropna()
    return [100.0 * (ranks <= k).mean() for k in k_values]


def plot_pubchem_topk_curves(model_dfs, rank_col, k_values):
    fig, ax = make_fig("square")
    for i, (label, df) in enumerate(model_dfs.items()):
        if rank_col not in df.columns:
            continue
        accs = topk_accuracy_curve(df, rank_col, k_values)
        color = model_color(label, fallback_index=i)
        ax.plot(k_values, accs, marker="o", label=label, color=color)
    ax.set_xlabel("k")
    ax.set_ylabel("Top-k accuracy (%)")
    ax.set_ylim(0, 100)
    ax.legend()
    return fig

In [ ]:
for mode in MODES:
    rank_col = f"rank_{mode}_cosine"
    dfs_with_mode = {
        label: df for label, df in model_dfs.items() if rank_col in df.columns
    }
    if not dfs_with_mode:
        continue
    fig = plot_pubchem_topk_curves(dfs_with_mode, rank_col, K_VALUES)
    fig.axes[0].set_xlabel("Top-k")
    save_fig(fig, f"retrieval_pubchem_global_topk_{mode}_scaffold", OUTPUT_DIR)
    plt.show()
    plt.close(fig)

## Top-k accuracy curve, zoomed inset (autofail)

Same as above but with a zoomed inset over low k, all models.

In [ ]:
def plot_pubchem_topk_curves_zoom(
    model_dfs, rank_col, k_values, inset_k_max=10, inset_y_max=10
):
    fig, ax = make_fig("square")
    axins = zoomed_inset_axes(
        ax,
        zoom=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.7),
        bbox_transform=ax.transAxes,
    )
    for i, (label, df) in enumerate(model_dfs.items()):
        if rank_col not in df.columns:
            continue
        accs = topk_accuracy_curve(df, rank_col, k_values)
        color = model_color(label, fallback_index=i)
        ax.plot(k_values, accs, marker="o", label=label, color=color)
        inset_k_values = [k for k in k_values if k <= inset_k_max]
        inset_accs = accs[: len(inset_k_values)]
        axins.plot(inset_k_values, inset_accs, marker="o", color=color)
    ax.set_xlabel("k")
    ax.set_ylabel("Top-k accuracy (%)")
    ax.set_ylim(0, 100)
    ax.legend(loc="upper right")
    axins.set_xlim(0, inset_k_max)
    axins.set_ylim(0, inset_y_max)
    axins.set_yticks([0, 5, 10])
    axins.set_box_aspect(1)
    mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5")
    return fig


rank_col = "rank_autofail_cosine"
dfs_with_mode = {
    label: df for label, df in model_dfs.items() if rank_col in df.columns
}
fig = plot_pubchem_topk_curves_zoom(dfs_with_mode, rank_col, K_VALUES)
save_fig(
    fig, "retrieval_pubchem_global_topk_autofail_scaffold_zoom", OUTPUT_DIR
)
plt.show()
plt.close(fig)

## Summary table (top-k accuracy, MRR, median rank)

Single run per model, so no ± CI column.

In [ ]:
rows = []
for mode in MODES:
    rank_col = f"rank_{mode}_cosine"
    for label, df in model_dfs.items():
        if rank_col not in df.columns:
            continue
        ranks = df[rank_col].dropna()
        row = {
            "mode": mode,
            "model": label,
            "n_queries": len(ranks),
        }
        for k in SUMMARY_K_VALUES:
            row[f"top-{k}"] = f"{100.0 * (ranks <= k).mean():.1f}"
        row["MRR"] = f"{(1.0 / ranks).mean():.4f}"
        row["median_rank"] = f"{ranks.median():.1f}"
        rows.append(row)

df_summary = pd.DataFrame(rows)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(
    OUTPUT_DIR / "retrieval_pubchem_global_summary_scaffold.csv", index=False
)
df_summary

## Export summary to LaTeX

In [ ]:
def export_pubchem_global_summary_latex(
    df,
    output_path="figures/retrieval_pubchem_global_scaffold/retrieval_pubchem_global_table_scaffold.tex",
):
    latex = df.to_latex(index=False, escape=False)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        f.write(latex)
    print(f"LaTeX table exported to {output_path}")
    print(latex)


export_pubchem_global_summary_latex(df_summary)

## MW-global: full-test-set candidate selection by molecular weight, no RI

Same models, same full test set as the top-level comparison above, but
candidates are restricted by molecular weight (symmetric window around the
query's own predicted highest-peak m/z) before ranking by cosine
similarity. Windows: ±5Da, ±10Da, ±80Da (matching the `mw_global=true`
scaffold reruns; the asymmetric [-10,+80]Da variant from the random-split
notebook was not run for scaffold).

In [ ]:
MW_GLOBAL_TAGS = {
    "Full-PubChem (no MW)": None,
    "MW ±80Da": "80",
    "MW ±10Da": "10",
    "MW ±5Da": "5",
}

MW_GLOBAL_MODEL_DIRS = {
    "ICICLE": "pubchem_retrieval_eval_icicle_scaffold_s1",
    "NEIMS": "pubchem_retrieval_eval_neims_scaffold_s1",
    "MassFormer": "pubchem_retrieval_eval_massformer_scaffold_s1",
}

In [ ]:
mw_global_dfs = {}  # {track_label: {model_label: df}}
for track_label, tag in MW_GLOBAL_TAGS.items():
    dfs = {}
    for label, dirname in MW_GLOBAL_MODEL_DIRS.items():
        if tag is None:
            path = RESULTS / dirname / "retrieval_global_per_query.tsv"
        else:
            path = RESULTS / dirname / f"retrieval_per_query_mw{tag}_all.tsv"
        if path.exists():
            dfs[label] = pd.read_csv(path, sep="\t")
            print(f"{track_label}/{label}: {len(dfs[label])} queries loaded")
        else:
            print(f"{track_label}/{label}: {path} not found, skipping")
    mw_global_dfs[track_label] = dfs

### Top-k accuracy vs. k, all MW tracks overlaid per model

One figure per (model x mode) so all candidate-selection strategies are
directly comparable on the same axes.

In [ ]:
def plot_global_tracks(track_dfs_by_label, model_label, rank_col, k_values):
    """track_dfs_by_label: {track_label: {model_label: df}}"""
    fig, ax = make_fig("default")
    for track_label, dfs in track_dfs_by_label.items():
        df = dfs.get(model_label)
        if df is None or rank_col not in df.columns:
            continue
        accs = topk_accuracy_curve(df, rank_col, k_values)
        ax.plot(k_values, accs, marker="o", label=track_label)
    ax.set_xlabel("Top-k")
    ax.set_ylabel("Top-k accuracy (%)")
    ax.set_ylim(0, 100)
    ax.legend()
    return fig


for model_label in MW_GLOBAL_MODEL_DIRS:
    for mode in MODES:
        rank_col = f"rank_{mode}_cosine"
        fig = plot_global_tracks(
            mw_global_dfs, model_label, rank_col, K_VALUES
        )
        fig.axes[0].set_title(
            f"{model_label} \u2014 {MODE_DISPLAY_NAMES.get(mode, mode)}"
        )
        save_fig(
            fig,
            f"retrieval_pubchem_global_mw_tracks_{model_label}_{mode}_scaffold",
            OUTPUT_DIR,
        )
        plt.show()
        plt.close(fig)

### MW-global summary table

In [ ]:
mw_global_rows = []
for track_label, dfs in mw_global_dfs.items():
    for mode in MODES:
        rank_col = f"rank_{mode}_cosine"
        for model_label, df in dfs.items():
            if rank_col not in df.columns:
                continue
            ranks = df[rank_col].dropna()
            row = {
                "track": track_label,
                "mode": mode,
                "model": model_label,
                "n_queries": len(ranks),
            }
            for k in SUMMARY_K_VALUES:
                row[f"top-{k}"] = f"{100.0 * (ranks <= k).mean():.1f}"
            row["mrr"] = f"{(1.0 / ranks).mean():.4f}"
            row["median_rank"] = ranks.median()
            mw_global_rows.append(row)

df_mw_global_summary = pd.DataFrame(mw_global_rows)
df_mw_global_summary.to_csv(
    OUTPUT_DIR / "retrieval_pubchem_global_mw_tracks_summary_scaffold.csv",
    index=False,
)
df_mw_global_summary

## Heavy-atom-global: full-test-set candidate selection by heavy-atom count, no RI

Same pattern as MW-global, but candidates are restricted by heavy-atom
count (query-side: model's own predicted heavy-atom count from its
predicted spectrum via `heavy_atom_predictor_random_split.joblib`;
candidate-side: RDKit-true heavy-atom count from each PubChem row's own
SMILES). Windows: +/-1, 2, 3, 6, 8 atoms.

Caveat: the heavy-atom predictor itself is trained only on the
**random**-split train set (no scaffold-split equivalent exists), so this
track mixes two different generalization questions -- the predictor's own
random-split-trained accuracy, applied to scaffold-split queries. Kept in
scope per explicit request, flagged here for the writeup.

In [ ]:
HA_GLOBAL_TAGS = {
    "Full-PubChem (no HA)": None,
    "HA \u00b11": "1",
    "HA \u00b12": "2",
    "HA \u00b13": "3",
    "HA \u00b16": "6",
    "HA \u00b18": "8",
}

HA_GLOBAL_MODEL_DIRS = {
    "ICICLE": "pubchem_retrieval_eval_icicle_scaffold_s1",
    "NEIMS": "pubchem_retrieval_eval_neims_scaffold_s1",
    "MassFormer": "pubchem_retrieval_eval_massformer_scaffold_s1",
}

ha_global_dfs = {}  # {track_label: {model_label: df}}
for track_label, tag in HA_GLOBAL_TAGS.items():
    dfs = {}
    for label, dirname in HA_GLOBAL_MODEL_DIRS.items():
        if tag is None:
            path = RESULTS / dirname / "retrieval_global_per_query.tsv"
        else:
            path = (
                RESULTS
                / dirname
                / f"retrieval_per_query_heavy_atom{tag}_all.tsv"
            )
        if path.exists():
            dfs[label] = pd.read_csv(path, sep="\t")
            print(f"{track_label}/{label}: {len(dfs[label])} queries loaded")
        else:
            print(f"{track_label}/{label}: {path} not found, skipping")
    ha_global_dfs[track_label] = dfs

### Top-k accuracy vs. k, all heavy-atom tracks overlaid per model

In [ ]:
for model_label in HA_GLOBAL_MODEL_DIRS:
    for mode in MODES:
        rank_col = f"rank_{mode}_cosine"
        fig = plot_global_tracks(
            ha_global_dfs, model_label, rank_col, K_VALUES
        )
        fig.axes[0].set_title(
            f"{model_label} \u2014 {MODE_DISPLAY_NAMES.get(mode, mode)}"
        )
        save_fig(
            fig,
            f"retrieval_pubchem_global_ha_tracks_{model_label}_{mode}_scaffold",
            OUTPUT_DIR,
        )
        plt.show()
        plt.close(fig)

### Heavy-atom-global summary table

In [ ]:
ha_global_rows = []
for track_label, dfs in ha_global_dfs.items():
    for mode in MODES:
        rank_col = f"rank_{mode}_cosine"
        for model_label, df in dfs.items():
            if rank_col not in df.columns:
                continue
            ranks = df[rank_col].dropna()
            row = {
                "track": track_label,
                "mode": mode,
                "model": model_label,
                "n_queries": len(ranks),
            }
            for k in SUMMARY_K_VALUES:
                row[f"top-{k}"] = f"{100.0 * (ranks <= k).mean():.1f}"
            row["mrr"] = f"{(1.0 / ranks).mean():.4f}"
            row["median_rank"] = ranks.median()
            ha_global_rows.append(row)

df_ha_global_summary = pd.DataFrame(ha_global_rows)
df_ha_global_summary.to_csv(
    OUTPUT_DIR / "retrieval_pubchem_global_ha_tracks_summary_scaffold.csv",
    index=False,
)
df_ha_global_summary